In [1]:
import requests
from bs4 import BeautifulSoup

URL = "https://webscraper.io/test-sites/e-commerce/static/computers/laptops"
response = requests.get(URL, headers={"User-Agent": "Mozilla/5.0 (learning project)"}, timeout=10)
response.raise_for_status()
print(response.status_code)

200


In [2]:
soup = BeautifulSoup(response.text, "html.parser")
cards = soup.select("div.thumbnail")
print(f"Found {len(cards)} products")

Found 6 products


In [3]:
print(cards[0].prettify())

<div class="card thumbnail" itemscope="" itemtype="https://schema.org/Product">
 <div class="product-wrapper card-body">
  <img alt="item" class="img-fluid card-img-top image img-responsive" itemprop="image" src="/images/test-sites/e-commerce/items/cart2.png"/>
  <div class="caption">
   <h4 class="price float-end card-title pull-right" itemprop="offers" itemscope="" itemtype="https://schema.org/Offer">
    <span itemprop="price">
     $416.99
    </span>
    <meta content="USD" itemprop="priceCurrency"/>
   </h4>
   <h4>
    <a class="title" href="/test-sites/e-commerce/static/product/31" itemprop="name" title="Packard 255 G2">
     Packard 255 G2
    </a>
   </h4>
   <p class="description card-text" itemprop="description">
    15.6", AMD E2-3800 1.3GHz, 4GB, 500GB, Windows 8.1
   </p>
  </div>
  <div class="ratings" itemprop="aggregateRating" itemscope="" itemtype="https://schema.org/AggregateRating">
   <p class="review-count float-end">
    <span itemprop="reviewCount">
     2
    

In [4]:
card = cards[0]

name = card.select_one("a.title")["title"]
price = card.select_one("h4.price").get_text(strip=True)
description = card.select_one("p.description").get_text(strip=True)
rating = card.select_one("p[data-rating]")["data-rating"]
reviews = card.select_one("span[itemprop='reviewCount']").get_text(strip=True)
url = "https://webscraper.io" + card.select_one("a.title")["href"]

print(name, price, description, rating, reviews, url, sep="\n")

Packard 255 G2
$416.99
15.6", AMD E2-3800 1.3GHz, 4GB, 500GB, Windows 8.1
2
2
https://webscraper.io/test-sites/e-commerce/static/product/31


In [5]:
#extract il products min page wahda
#pandas = python library for working with tables of data

import pandas as pd

products = []
for card in cards:
    product = {
        "name": card.select_one("a.title")["title"],
        "price": card.select_one("h4.price").get_text(strip=True),
        "description": card.select_one("p.description").get_text(strip=True),
        "rating": card.select_one("p[data-rating]")["data-rating"],
        "reviews": card.select_one("span[itemprop='reviewCount']").get_text(strip=True),
        "url": "https://webscraper.io" + card.select_one("a.title")["href"],
    }
    products.append(product)

df = pd.DataFrame(products)
df

,name,price,description,rating,reviews,url
0,Packard 255 G2,$416.99,"15.6"", AMD E2-3800 1.3GHz, 4GB, 500GB, Windows...",2,2,https://webscraper.io/test-sites/e-commerce/st...
1,Aspire E1-510,$306.99,"15.6"", Pentium N3520 2.16GHz, 4GB, 500GB, Linux",3,2,https://webscraper.io/test-sites/e-commerce/st...
2,ThinkPad T540p,$1178.99,"15.6"", Core i5-4200M, 4GB, 500GB, Win7 Pro 64bit",1,2,https://webscraper.io/test-sites/e-commerce/st...
3,ProBook,$739.99,"14"", Core i5 2.6GHz, 4GB, 500GB, Win7 Pro 64bit",4,8,https://webscraper.io/test-sites/e-commerce/st...
4,ThinkPad X240,$1311.99,"12.5"", Core i5-4300U, 8GB, 240GB SSD, Win7 Pro...",3,12,https://webscraper.io/test-sites/e-commerce/st...
5,Aspire E1-572G,$581.99,"15.6"", Core i5-4200U, 8GB, 1TB, Radeon R7 M265...",1,2,https://webscraper.io/test-sites/e-commerce/st...


In [6]:
#Func extract products min specific page

def scrape_page(page_number):
    url = f"https://webscraper.io/test-sites/e-commerce/static/computers/laptops?page={page_number}"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (learning project)"}, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    cards = soup.select("div.thumbnail")

    products = []
    for card in cards:
        products.append({
            "name": card.select_one("a.title")["title"],
            "price": card.select_one("h4.price").get_text(strip=True),
            "description": card.select_one("p.description").get_text(strip=True),
            "rating": card.select_one("p[data-rating]")["data-rating"],
            "reviews": card.select_one("span[itemprop='reviewCount']").get_text(strip=True),
            "url": "https://webscraper.io" + card.select_one("a.title")["href"],
        })
    return products

#testtt fel page 2
page2 = scrape_page(2)
print(len(page2), "products")
print(page2[0]["name"])

6 products
ThinkPad Yoga


In [7]:
import time

all_products = []
page = 1

while page <= 30:                       # safety limit
    try:
        products = scrape_page(page)
    except requests.HTTPError:          # page doesn't exist -> we're past the end
        break
    if not products:                    # page exists but is empty -> stop too
        break

    all_products.extend(products)
    print(f"Page {page}: {len(products)} products (total: {len(all_products)})")
    page += 1
    time.sleep(1)                       # wait 1 second between requests

df = pd.DataFrame(all_products)
print(df.shape)
df.head()

Page 1: 6 products (total: 6)
Page 2: 6 products (total: 12)
Page 3: 6 products (total: 18)
Page 4: 6 products (total: 24)
Page 5: 6 products (total: 30)
Page 6: 6 products (total: 36)
Page 7: 6 products (total: 42)
Page 8: 6 products (total: 48)
Page 9: 6 products (total: 54)
Page 10: 6 products (total: 60)
Page 11: 6 products (total: 66)
Page 12: 6 products (total: 72)
Page 13: 6 products (total: 78)
Page 14: 6 products (total: 84)
Page 15: 6 products (total: 90)
Page 16: 6 products (total: 96)
Page 17: 6 products (total: 102)
Page 18: 6 products (total: 108)
Page 19: 6 products (total: 114)
Page 20: 3 products (total: 117)
(117, 6)


,name,price,description,rating,reviews,url
0,Packard 255 G2,$416.99,"15.6"", AMD E2-3800 1.3GHz, 4GB, 500GB, Windows...",2,2,https://webscraper.io/test-sites/e-commerce/st...
1,Aspire E1-510,$306.99,"15.6"", Pentium N3520 2.16GHz, 4GB, 500GB, Linux",3,2,https://webscraper.io/test-sites/e-commerce/st...
2,ThinkPad T540p,$1178.99,"15.6"", Core i5-4200M, 4GB, 500GB, Win7 Pro 64bit",1,2,https://webscraper.io/test-sites/e-commerce/st...
3,ProBook,$739.99,"14"", Core i5 2.6GHz, 4GB, 500GB, Win7 Pro 64bit",4,8,https://webscraper.io/test-sites/e-commerce/st...
4,ThinkPad X240,$1311.99,"12.5"", Core i5-4300U, 8GB, 240GB SSD, Win7 Pro...",3,12,https://webscraper.io/test-sites/e-commerce/st...


In [8]:
import os

print(df.shape)                      # (rows, columns)
print(df["url"].nunique())           # number of unique products

os.makedirs("data", exist_ok=True)
df.to_csv("data/laptops_raw.csv", index=False)
print("Saved")

(117, 6)
117
Saved


In [9]:
#Next etape clean il data
df_clean = df.copy()

#Price: "$416.99" -> 416.99
df_clean["price"] = df_clean["price"].str.replace("$", "", regex=False).astype(float)

#Rating and reviews: text -> integers
df_clean["rating"] = df_clean["rating"].astype(int)
df_clean["reviews"] = df_clean["reviews"].astype(int)

#Remove duplicate products(2 rows with same url = same product)
df_clean = df_clean.drop_duplicates(subset="url")

#Save the cleaned version
df_clean.to_csv("data/laptops_clean.csv", index=False)

print(df_clean.dtypes) #shows the data type of each column
print(df_clean.shape)
df_clean.sort_values("price").head()

name               str
price          float64
description        str
rating           int64
reviews          int64
url                str
dtype: object
(117, 6)


,name,price,description,rating,reviews,url
29,Asus VivoBook X441NA-GA190,295.99,"Asus VivoBook X441NA-GA190 Chocolate Black, 14...",3,14,https://webscraper.io/test-sites/e-commerce/st...
30,Prestigio SmartBook 133S Dark Grey,299.00,"Prestigio SmartBook 133S Dark Grey, 13.3"" FHD ...",2,8,https://webscraper.io/test-sites/e-commerce/st...
31,Prestigio SmartBook 133S Gold,299.00,"Prestigio SmartBook 133S Gold, 13.3"" FHD IPS, ...",4,12,https://webscraper.io/test-sites/e-commerce/st...
1,Aspire E1-510,306.99,"15.6"", Pentium N3520 2.16GHz, 4GB, 500GB, Linux",3,2,https://webscraper.io/test-sites/e-commerce/st...
32,Lenovo V110-15IAP,321.94,"Lenovo V110-15IAP, 15.6"" HD, Celeron N3350 1.1...",3,5,https://webscraper.io/test-sites/e-commerce/st...


In [10]:
import re

def extract_specs(desc):
    screen = re.search(r'([\d.]+)"', desc)
    sizes = re.findall(r'(\d+)\s*(GB|TB)', desc)   

    def to_gb(value, unit):
        return int(value) * (1000 if unit == "TB" else 1)

    ram = to_gb(*sizes[0]) if len(sizes) > 0 else None
    storage = to_gb(*sizes[1]) if len(sizes) > 1 else None
    return pd.Series({
        "screen_inch": float(screen.group(1)) if screen else None,
        "ram_gb": ram,
        "storage_gb": storage,
    })

specs = df_clean["description"].apply(extract_specs)
df_clean = pd.concat([df_clean, specs], axis=1)

print(df_clean[["screen_inch", "ram_gb", "storage_gb"]].isna().sum())   # 9adeh adna min missing valeur f column ram,storage,screen..
df_clean.to_csv("data/laptops_clean.csv", index=False)
df_clean[df_clean["ram_gb"] >= 8].sort_values("price").head()

screen_inch    0
ram_gb         0
storage_gb     0
dtype: int64


,name,price,description,rating,reviews,url,screen_inch,ram_gb,storage_gb
43,Lenovo V110-15ISK,409.63,"Lenovo V110-15ISK, 15.6"" HD, Core i3-6006U, 8G...",3,9,https://webscraper.io/test-sites/e-commerce/st...,15.6,8.0,128.0
5,Aspire E1-572G,581.99,"15.6"", Core i5-4200U, 8GB, 1TB, Radeon R7 M265...",1,2,https://webscraper.io/test-sites/e-commerce/st...,15.6,8.0,1000.0
17,Acer Aspire A515-51-5654,679.00,"Acer Aspire A515-51-5654, Black, 15.6"", FHD, C...",2,9,https://webscraper.io/test-sites/e-commerce/st...,15.6,8.0,256.0
18,Dell Inspiron 15,679.00,"Dell Inspiron 15 (5567) Fog Gray, 15.6"" FHD, C...",2,7,https://webscraper.io/test-sites/e-commerce/st...,15.6,8.0,1000.0
19,Asus VivoBook S14,729.00,"Asus VivoBook S14 (S406UA-BV041T) Starry Grey,...",1,2,https://webscraper.io/test-sites/e-commerce/st...,14.0,8.0,256.0


In [11]:
# preparation mt3 data lel chatbot
df_clean = pd.read_csv("data/laptops_clean.csv")

columns = ["name", "price", "rating", "reviews", "screen_inch", "ram_gb", "storage_gb", "description"]
context = df_clean[columns].to_csv(index=False)

print(len(context), "characters")
print(context[:500])

17309 characters
name,price,rating,reviews,screen_inch,ram_gb,storage_gb,description
Packard 255 G2,416.99,2,2,15.6,4.0,500.0,"15.6"", AMD E2-3800 1.3GHz, 4GB, 500GB, Windows 8.1"
Aspire E1-510,306.99,3,2,15.6,4.0,500.0,"15.6"", Pentium N3520 2.16GHz, 4GB, 500GB, Linux"
ThinkPad T540p,1178.99,1,2,15.6,4.0,500.0,"15.6"", Core i5-4200M, 4GB, 500GB, Win7 Pro 64bit"
ProBook,739.99,4,8,14.0,4.0,500.0,"14"", Core i5 2.6GHz, 4GB, 500GB, Win7 Pro 64bit"
ThinkPad X240,1311.99,3,12,12.5,8.0,240.0,"12.5"", Core i5-430
